# R6 — Conversation memory (summary)

Members ask follow-ups like "and for group plans?" that mean nothing alone. This assistant carries a **short running summary** of the chat (not the whole history), uses it to **restate each follow-up as a stand-alone question** before answering, and **traces the chat with MLflow** so support staff can inspect it later.

Cells:
1. The R5 chain plus memory: running-summary update, follow-up rewrite to a stand-alone question, one MLflow trace over the whole chat.
2. Replay the client's scripted 3-turn chat word for word: turn 3 ("And for group plans?") must cite a `03_Waiting_Periods_Group_Plans.md` chunk first.
3. Write `evidence/r6_memory.json` (turns ×3, summary ≥40 chars, trace_id) and verify it.

Answers come only from the retrieved policy sections (vector search top-5 on `claims.genai.policy_chunks_idx`); unanswerable → exactly `not in policy`.

In [0]:
# Step 1 — the R5 chain + conversation memory.
# Retrieval/generation are identical to R5; memory adds a running summary, a
# follow-up rewrite to a stand-alone question, and one MLflow trace for the chat.

import re, time, requests
import mlflow
from databricks.sdk import WorkspaceClient

_w = WorkspaceClient()
_HOST = _w.config.host if _w.config.host.startswith("http") else "https://" + _w.config.host
_AUTH = _w.config.authenticate()["Authorization"]
LLM_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"
NOT_IN_POLICY = "not in policy"


def _chat(endpoint, system, user, max_tokens=600):
    """One chat-completion call; retries with backoff on 429 rate limits / 5xx."""
    payload = {"messages": [{"role": "system", "content": system},
                            {"role": "user", "content": user}],
               "max_tokens": max_tokens, "temperature": 0.0}
    headers = {"Authorization": _AUTH, "Content-Type": "application/json"}
    for attempt in range(6):
        r = requests.post(f"{_HOST}/serving-endpoints/{endpoint}/invocations",
                          headers=headers, json=payload, timeout=180)
        if r.status_code == 200:
            return r.json()["choices"][0]["message"]["content"]
        if r.status_code == 429 or r.status_code >= 500:
            time.sleep(min(2 ** attempt * 3, 30))   # rate limited - back off and retry
            continue
        r.raise_for_status()
    r.raise_for_status()


def _retrieve(question, k=5):
    qlit = question.replace("'", "''")
    df = spark.sql(f"""
      SELECT vs.chunk_id
      FROM (SELECT '{qlit}' AS qtxt) g,
      LATERAL vector_search(index => 'claims.genai.policy_chunks_idx',
                            query_text => g.qtxt, num_results => {k}) vs
    """)
    return [row.chunk_id for row in df.collect()]


_chunk_cache = {}
def _chunk_text(chunk_id):
    if chunk_id not in _chunk_cache:
        _chunk_cache[chunk_id] = spark.sql(
            "SELECT text FROM claims.genai.policy_chunks WHERE chunk_id = '{}'".format(
                chunk_id.replace("'", "''"))
        ).collect()[0][0]
    return _chunk_cache[chunk_id]


SYSTEM_PROMPT = (
    "You are the Saffron Shield policy binder assistant for insurance adjusters. "
    "Answer the question using ONLY the policy sections provided in the context. "
    "If the context does not contain the information needed to answer, reply with exactly: "
    "not in policy\n"
    "and nothing else. Never use the phrase 'not in policy' unless the context truly cannot "
    "answer the question. Never use outside knowledge. Be concise and factual; quote the "
    "relevant rule from the sections."
)


def _answer_from_sections(standalone_question, k=5):
    """Grounded answer + citations for a stand-alone question (the R5 chain)."""
    cited = _retrieve(standalone_question, k)
    parts = [f"[{i + 1}] {cid}\n{_chunk_text(cid)}" for i, cid in enumerate(cited)]
    context = "\n\n---\n\n".join(parts)
    answer = _chat(LLM_ENDPOINT, SYSTEM_PROMPT, f"Context:\n{context}\n\nQuestion: {standalone_question}").strip()
    if re.fullmatch(r"not[ .]*in[ .]*policy[.!\s]*", answer, re.IGNORECASE):
        answer = NOT_IN_POLICY
    return {"answer": answer, "cited_chunk_ids": cited}


def rewrite_standalone(running_summary, follow_up):
    """Restate a follow-up as a stand-alone question using the running summary only."""
    if not running_summary:
        return follow_up
    out = _chat(
        LLM_ENDPOINT,
        "You rewrite follow-up questions so they stand alone. Use ONLY the conversation "
        "summary provided. Replace pronouns and vague references (that, it, them, also) with "
        "what they refer to. Do not answer the question. Reply with the rewritten question "
        "only, no quotes, no explanation.",
        f"Conversation summary:\n{running_summary}\n\nFollow-up question: {follow_up}",
        max_tokens=200,
    ).strip().strip('"')
    return out or follow_up


def update_summary(running_summary, turn_record):
    """Short running summary of the chat so far (NOT the whole history)."""
    return _chat(
        LLM_ENDPOINT,
        "Maintain a SHORT running summary of a policy support chat (max 3 sentences). "
        "Keep only what later turns could refer back to: the topic, the plan type being "
        "discussed, and facts already established. Reply with the summary text only.",
        f"Current summary:\n{running_summary or '(empty)'}\n\nNew turn:\n"
        f"User asked: {turn_record['question']}\n"
        f"(stand-alone: {turn_record['standalone_question']})\n"
        f"Assistant answered: {turn_record['answer']}",
        max_tokens=250,
    ).strip()


@mlflow.trace(name="r6_chat_with_memory")
def run_chat(turns):
    """Replay the scripted chat; one MLflow trace spans the whole conversation."""
    summary = ""
    records = []
    for i, question in enumerate(turns, start=1):
        standalone = rewrite_standalone(summary, question) if i > 1 else question
        res = _answer_from_sections(standalone)
        rec = {"turn": i, "question": question, "standalone_question": standalone, **res}
        records.append(rec)
        summary = update_summary(summary, rec)
        print(f"--- turn {i} ---")
        print("question          :", question)
        print("standalone_question:", standalone)
        print("answer            :", res["answer"][:180].replace("\n", " "))
        print("cited_chunk_ids   :", res["cited_chunk_ids"][:2], "...")
        print("summary so far    :", summary[:200].replace("\n", " "))
    return records, summary


print("chain + memory ready; mlflow", mlflow.__version__)


In [0]:
# Step 2 — replay the client's scripted chat, word for word.

SCRIPTED_TURNS = [
    "What is the pre-existing disease waiting period on an individual plan?",
    "Does moving from Silver to Gold restart that count?",
    "And for group plans?",
]

records, summary = run_chat(SCRIPTED_TURNS)
trace_id = mlflow.get_last_active_trace_id()
print("\ntrace_id:", trace_id)


In [0]:
# Step 3 — write evidence/r6_memory.json and verify it against the spec.

import json, os

EVIDENCE_DIR = "/Workspace/Users/deepaksinghal@cyntexa.com/-DE---Scenario-based-Assignment-1/Gen AI - Paper 4 Scenario Based Assignment/evidence"
os.makedirs(EVIDENCE_DIR, exist_ok=True)

memory_doc = {
    "turns": [{k: r[k] for k in ("turn", "question", "standalone_question", "answer", "cited_chunk_ids")}
              for r in records],
    "summary": summary,
    "trace_id": trace_id,
}
path = os.path.join(EVIDENCE_DIR, "r6_memory.json")
with open(path, "w") as f:
    json.dump(memory_doc, f, indent=2)
print("wrote", path)

# verification against the spec
turns, doc_summary, doc_trace = memory_doc["turns"], memory_doc["summary"], memory_doc["trace_id"]
checks = [
    ("3 turn objects with the exact keys", len(turns) == 3 and
     all(set(t) == {"turn", "question", "standalone_question", "answer", "cited_chunk_ids"} for t in turns)),
    ("turn numbers 1,2,3 and scripted questions word for word",
     [t["turn"] for t in turns] == [1, 2, 3] and
     [t["question"] for t in turns] == SCRIPTED_TURNS),
    ("cited_chunk_ids are lists of bare chunk ids",
     all(isinstance(t["cited_chunk_ids"], list) and len(t["cited_chunk_ids"]) >= 1 and
         all("#" in c for c in t["cited_chunk_ids"]) for t in turns)),
    ("summary is text of at least 40 characters", isinstance(doc_summary, str) and len(doc_summary.strip()) >= 40),
    ("trace_id non-empty", isinstance(doc_trace, str) and len(doc_trace) > 0),
    ("turn 3 cites a 03_Waiting_Periods_Group_Plans.md chunk FIRST",
     turns[2]["cited_chunk_ids"][0].startswith("03_Waiting_Periods_Group_Plans.md")),
]
for name, ok in checks:
    print(("PASS" if ok else "FAIL"), "-", name)
assert all(ok for _, ok in checks), "r6_memory.json verification failed"
